# Remove Background & Foreground Enhancement

Implementasi sederhana untuk:
1. **Remove Background** - Menghapus background dari clothing images
2. **Foreground Enhancement** - Meningkatkan kualitas objek utama (clothing)
3. **Low Quality Image Handling** - Optimasi untuk image low resolution/quality

## 1. Install Dependencies

In [ ]:
!pip install rembg opencv-python pillow scikit-image -q

print(" Dependencies installed")

## 2. Import Libraries

In [ ]:
import cv2
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter
from rembg import remove
import matplotlib.pyplot as plt
import os
import pandas as pd
from skimage import exposure, filters
from skimage.restoration import denoise_bilateral

print("✓ Libraries imported")

## 3. Load Sample Images

In [ ]:
train_dir = 'train/train/'
train_df = pd.read_csv('train.csv')

def load_image(image_id, image_dir):
    """Load image dengan berbagai format"""
    for ext in ['.jpg', '.png', '.jpeg']:
        image_path = os.path.join(image_dir, f"{image_id}{ext}")
        if os.path.exists(image_path):
            return Image.open(image_path).convert('RGB')
    return None

# Load 3 sample images
sample_ids = train_df.sample(3, random_state=42)['id'].tolist()
sample_images = []

for img_id in sample_ids:
    img = load_image(img_id, train_dir)
    if img is not None:
        sample_images.append((img_id, img))

print(f"✓ Loaded {len(sample_images)} sample images")
print(f"Sample IDs: {[img_id for img_id, _ in sample_images]}")

## 4. Foreground Enhancement Functions

Fungsi untuk meningkatkan kualitas gambar low resolution/quality.

In [ ]:
def enhance_image_quality(image):
    """
    Meningkatkan kualitas gambar low resolution/quality
    - Sharpening untuk detail lebih tajam
    - Contrast enhancement
    - Brightness adjustment
    - Denoising untuk mengurangi noise
    """
    # Convert PIL to numpy array
    img_array = np.array(image)
    
    # 1. Denoise (reduce noise for low quality images)
    denoised = denoise_bilateral(img_array, sigma_color=0.05, sigma_spatial=15, 
                                  channel_axis=-1)
    denoised = (denoised * 255).astype(np.uint8)
    img_pil = Image.fromarray(denoised)
    
    # 2. Contrast enhancement
    enhancer = ImageEnhance.Contrast(img_pil)
    img_pil = enhancer.enhance(1.3)
    
    # 3. Sharpness enhancement
    enhancer = ImageEnhance.Sharpness(img_pil)
    img_pil = enhancer.enhance(1.8)
    
    # 4. Brightness adjustment (slight)
    enhancer = ImageEnhance.Brightness(img_pil)
    img_pil = enhancer.enhance(1.1)
    
    # 5. Unsharp mask for extra sharpness
    img_pil = img_pil.filter(ImageFilter.UnsharpMask(radius=2, percent=150, threshold=3))
    
    return img_pil

def adaptive_histogram_equalization(image):
    """
    CLAHE (Contrast Limited Adaptive Histogram Equalization)
    Meningkatkan kontras lokal untuk detail lebih baik
    """
    img_array = np.array(image)
    
    # Convert to LAB color space
    img_lab = cv2.cvtColor(img_array, cv2.COLOR_RGB2LAB)
    
    # Apply CLAHE to L channel
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    img_lab[:, :, 0] = clahe.apply(img_lab[:, :, 0])
    
    # Convert back to RGB
    img_enhanced = cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)
    
    return Image.fromarray(img_enhanced)

def super_resolution_upscale(image, scale=2):
    """
    Upscale image menggunakan bicubic interpolation
    Untuk low resolution images
    """
    width, height = image.size
    new_size = (width * scale, height * scale)
    
    # Bicubic interpolation untuk hasil lebih smooth
    upscaled = image.resize(new_size, Image.BICUBIC)
    
    return upscaled

print("✓ Enhancement functions defined")

## 5. Background Removal Functions

Fungsi untuk menghapus background dari clothing images.

In [ ]:
def remove_background_rembg(image):
    """
    Remove background menggunakan library rembg (AI-based)
    Menggunakan pretrained model u2net
    """
    # Convert PIL to bytes
    output = remove(image)
    return output

def remove_background_grabcut(image):
    """
    Remove background menggunakan GrabCut algorithm (OpenCV)
    Lebih cepat tapi kurang akurat untuk complex backgrounds
    """
    img_array = np.array(image)
    
    # Create mask
    mask = np.zeros(img_array.shape[:2], np.uint8)
    
    # Define rectangle around foreground (assume center object)
    height, width = img_array.shape[:2]
    rect = (int(width*0.1), int(height*0.1), int(width*0.8), int(height*0.8))
    
    # GrabCut algorithm
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)
    
    cv2.grabCut(img_array, mask, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_RECT)
    
    # Create binary mask
    mask2 = np.where((mask == 2) | (mask == 0), 0, 1).astype('uint8')
    
    # Apply mask
    result = img_array * mask2[:, :, np.newaxis]
    
    # White background
    white_bg = np.ones_like(img_array) * 255
    result = np.where(mask2[:, :, np.newaxis] == 1, result, white_bg)
    
    return Image.fromarray(result.astype(np.uint8))

def apply_white_background(image_with_alpha):
    """
    Apply white background untuk image dengan transparency
    """
    if image_with_alpha.mode != 'RGBA':
        return image_with_alpha
    
    # Create white background
    white_bg = Image.new('RGB', image_with_alpha.size, (255, 255, 255))
    
    # Paste image with alpha
    white_bg.paste(image_with_alpha, mask=image_with_alpha.split()[3])
    
    return white_bg

print("✓ Background removal functions defined")

## 6. Complete Pipeline

Pipeline lengkap: Enhancement → Background Removal

In [ ]:
def process_image_complete(image, method='rembg'):
    """
    Complete pipeline untuk low quality images:
    1. Enhance image quality (denoise, sharpen, contrast)
    2. Remove background
    3. Apply white background
    
    Args:
        image: PIL Image
        method: 'rembg' (AI-based, accurate) atau 'grabcut' (faster, less accurate)
    
    Returns:
        enhanced_image, nobg_image, final_image
    """
    # Step 1: Enhance image quality
    print("  → Enhancing image quality...")
    enhanced = enhance_image_quality(image)
    
    # Step 2: Remove background
    print(f"  → Removing background ({method})...")
    if method == 'rembg':
        nobg = remove_background_rembg(enhanced)
    else:
        nobg = remove_background_grabcut(enhanced)
    
    # Step 3: Apply white background
    print("  → Applying white background...")
    final = apply_white_background(nobg)
    
    return enhanced, nobg, final

print("✓ Complete pipeline defined")

## 7. Process Sample Images

Proses semua sample images dengan pipeline lengkap.

In [ ]:
results = []

for img_id, img in sample_images:
    print(f"\nProcessing image {img_id}...")
    enhanced, nobg, final = process_image_complete(img, method='rembg')
    results.append({
        'id': img_id,
        'original': img,
        'enhanced': enhanced,
        'nobg': nobg,
        'final': final
    })

print(f"\n✓ Processed {len(results)} images successfully")

## 8. Visualisasi: Complete Pipeline

Tampilkan hasil dari setiap tahap: Original → Enhanced → No Background → Final

In [ ]:
fig, axes = plt.subplots(len(results), 4, figsize=(20, 5*len(results)))
if len(results) == 1:
    axes = [axes]

for idx, result in enumerate(results):
    img_id = result['id']
    
    # Column 1: Original
    axes[idx][0].imshow(result['original'])
    axes[idx][0].set_title(f'Original Image\nID: {img_id}', fontsize=12, fontweight='bold')
    axes[idx][0].axis('off')
    
    # Column 2: Enhanced
    axes[idx][1].imshow(result['enhanced'])
    axes[idx][1].set_title('Enhanced\n(Denoise + Sharpen + Contrast)', 
                           fontsize=12, fontweight='bold', color='blue')
    axes[idx][1].axis('off')
    
    # Column 3: No Background (with transparency)
    axes[idx][2].imshow(result['nobg'])
    axes[idx][2].set_title('Background Removed\n(rembg AI)', 
                           fontsize=12, fontweight='bold', color='green')
    axes[idx][2].axis('off')
    
    # Column 4: Final (white background)
    axes[idx][3].imshow(result['final'])
    axes[idx][3].set_title('Final Result\n(White Background)', 
                           fontsize=12, fontweight='bold', color='red')
    axes[idx][3].axis('off')

plt.tight_layout()
plt.savefig('remove_bg_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Visualization saved: remove_bg_pipeline.png")

## 9. Visualisasi: Before vs After

Perbandingan sederhana Original vs Final Result

In [ ]:
fig, axes = plt.subplots(len(results), 2, figsize=(14, 6*len(results)))
if len(results) == 1:
    axes = [axes]

for idx, result in enumerate(results):
    img_id = result['id']
    
    # Before: Original
    axes[idx][0].imshow(result['original'])
    axes[idx][0].set_title(f'BEFORE\nOriginal Image (ID: {img_id})', 
                           fontsize=14, fontweight='bold', color='red')
    axes[idx][0].axis('off')
    axes[idx][0].set_facecolor('#ffeeee')
    
    # After: Final
    axes[idx][1].imshow(result['final'])
    axes[idx][1].set_title(f'AFTER\nEnhanced + Background Removed', 
                           fontsize=14, fontweight='bold', color='green')
    axes[idx][1].axis('off')
    axes[idx][1].set_facecolor('#eeffee')

plt.tight_layout()
plt.savefig('remove_bg_before_after.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Visualization saved: remove_bg_before_after.png")

## 10. Comparison: rembg vs GrabCut

Bandingkan kedua metode background removal.